Scenario 1: Retail Inventory Analysis

Prompt:
Act as a senior Python data analyst. Using a DataFrame with product_id, category, stock_level, last_restock_date, sales_last_30_days, supplier_lead_time, and unit_cost, calculate inventory turnover, identify slow-moving items, predict stockout risk, validate the data, and create visualizations using pandas and matplotlib.

Follow-up Prompt:
Review the code for missing values, invalid dates, negative values, division-by-zero errors, and improve the visualizations.

Final Solution Summary:
The final code converts dates with pd.to_datetime(), validates required columns, calculates daily sales, inventory turnover, days of stock remaining, slow-moving items, and stockout risk. Bar charts are used to show product turnover and category performance.

Discussion:
Date calculations were improved by converting restock dates to datetime and calculating days since restocking. Bar charts were suggested for visualization. Error handling included missing-column checks, invalid values, and division-by-zero protection.

Scenario 2: Website Analytics Debug

Prompt:
Act as a senior Python developer. Debug the provided engagement code that produces bounce rates above 100% and negative session times. Add validation, calculate metrics at session level, handle invalid durations, and create visualizations.

Follow-up Prompt:
Check for missing values, negative durations, duplicates, invalid page views, and explain the fixes.

Final Solution Summary:
The corrected code groups data by session_id, calculates total page views and duration per session, defines a bounce as a session with one page view, removes invalid durations, and calculates correct device-level metrics.

Discussion:
Providing the broken code helped the AI identify the problem quickly. Validation included missing values, duplicates, negative durations, and invalid page views. Time handling improved by calculating total duration per session before finding the average.

Scenario 3: Customer Segmentation Query

Prompt:
Act as a senior SQL analyst. Using the provided tables, identify users active in the last 30 days, calculate customer spending, select the top 20% of customers by spending, and return their preference information. Use CTEs and explain the query.

Follow-up Prompt:
Optimize the query, explain the percentile method, handle NULL values, and suggest useful indexes.

Final Solution Summary:
The SQL uses CTEs to separate active users, customer spending, rankings, and preferences. CUME_DIST() or NTILE(5) can be used to identify the top 20% of customers.

Discussion:
Percentiles were handled with SQL window functions. Active users were filtered using the last 30 days. CTEs made the query easier to read, debug, and optimize.

Overall Discussion

More detailed prompts produced better results. Role-based prompting, clear requirements, and follow-up prompts helped improve validation, debugging, and optimization. The main lesson was that AI-generated code should always be reviewed and refined before use.

Scenario 1: Retail Inventory Analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Convert date
df['last_restock_date'] = pd.to_datetime(
    df['last_restock_date'],
    errors='coerce'
)


# Average daily sales
df['daily_sales'] = df['sales_last_30_days'] / 30


# Inventory turnover
df['inventory_turnover'] = np.where(
    df['stock_level'] > 0,
    df['sales_last_30_days'] / df['stock_level'],
    np.nan
)


# Days inventory remaining
df['days_inventory_remaining'] = np.where(
    df['daily_sales'] > 0,
    df['stock_level'] / df['daily_sales'],
    np.inf
)


# Slow-moving items
df['slow_moving'] = df['inventory_turnover'] < 0.25


# Stockout risk
df['stockout_risk'] = (
    df['days_inventory_remaining'] <= df['supplier_lead_time']
)


# Show results
display(df)


# Visualization
df.groupby('category')['inventory_turnover'].mean().plot(
    kind='bar',
    figsize=(8, 5)
)


plt.title('Average Inventory Turnover by Category')
plt.xlabel('Category')
plt.ylabel('Turnover Rate')
plt.tight_layout()
plt.show()

Scenario 2: Website Analytics Debug
import pandas as pd
import matplotlib.pyplot as plt


def analyze_user_engagement(logs_df):


    df = logs_df.copy()


    # Convert numeric columns
    df['duration'] = pd.to_numeric(
        df['duration'],
        errors='coerce'
    )


    df['page_views'] = pd.to_numeric(
        df['page_views'],
        errors='coerce'
    )


    # Remove invalid data
    df = df.dropna(
        subset=['session_id', 'duration', 'page_views']
    )


    df = df[
        (df['duration'] >= 0) &
        (df['page_views'] >= 0)
    ]


    # Aggregate by session
    session_metrics = df.groupby(
        'session_id'
    ).agg(
        total_page_views=('page_views', 'sum'),
        total_duration=('duration', 'sum'),
        device_type=('device_type', 'first')
    ).reset_index()


    # Bounce session
    session_metrics['is_bounce'] = (
        session_metrics['total_page_views'] == 1
    )


    # Overall metrics
    metrics = {
        'bounce_rate':
            session_metrics['is_bounce'].mean() * 100,


        'avg_session_time':
            session_metrics['total_duration'].mean(),


        'pages_per_session':
            session_metrics['total_page_views'].mean()
    }


    # Device metrics
    device_metrics = session_metrics.groupby(
        'device_type'
    ).agg(
        sessions=('session_id', 'count'),
        avg_duration=('total_duration', 'mean'),
        avg_pages=('total_page_views', 'mean')
    )


    return metrics, session_metrics, device_metrics




metrics, session_metrics, device_metrics = (
    analyze_user_engagement(logs_df)
)


print(metrics)
display(device_metrics)


# Visualization
device_metrics['avg_duration'].plot(
    kind='bar',
    figsize=(8, 5)
)


plt.title('Average Session Duration by Device')
plt.xlabel('Device Type')
plt.ylabel('Average Duration')
plt.tight_layout()
plt.show()

Scenario 3: Customer Segmentation SQL
WITH active_users AS (
    SELECT
        user_id,
        account_type,
        feature_usage_count
    FROM user_activity
    WHERE last_login_date >= CURRENT_DATE - INTERVAL '30 days'
),


customer_spending AS (
    SELECT
        user_id,
        SUM(COALESCE(amount, 0)) AS total_spending
    FROM transactions
    GROUP BY user_id
),


ranked_customers AS (
    SELECT
        user_id,
        total_spending,
        CUME_DIST() OVER (
            ORDER BY total_spending DESC
        ) AS spending_rank
    FROM customer_spending
),


high_value_users AS (
    SELECT
        a.user_id,
        a.account_type,
        a.feature_usage_count,
        r.total_spending
    FROM active_users a
    JOIN ranked_customers r
        ON a.user_id = r.user_id
    WHERE r.spending_rank <= 0.20
)


SELECT
    h.user_id,
    h.account_type,
    h.feature_usage_count,
    h.total_spending,
    p.communication_preference,
    p.interface_theme,
    p.notification_settings
FROM high_value_users h


LEFT JOIN user_preferences p
    ON h.user_id = p.user_id


ORDER BY h.total_spending DESC;